In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torchattacks
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve
from scipy.spatial.distance import mahalanobis
from scipy.linalg import inv
import warnings
warnings.filterwarnings('ignore')

# ----------------------------
# 1. Настройки эксперимента
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.001
EPS_BIM = 0.2
BIM_STEPS = 10
BIM_ALPHA = 0.02

# ----------------------------
# 2. Подготовка данных
# ----------------------------
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
trainset = torchvision.datasets.FashionMNIST('./data', train=True, download=True, transform=transform)
testset  = torchvision.datasets.FashionMNIST('./data', train=False, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
testloader  = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ----------------------------
# 3. Архитектуры моделей
# ----------------------------
class Model_Best(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        with torch.no_grad():
            self.feat_size = self.features(torch.zeros(1, 1, 28, 28)).numel()
        self.fc1 = nn.Linear(self.feat_size, 512)
        self.fc2 = nn.Linear(512, 10)
        self.relu = nn.ReLU()
        self.feature_maps = {}

    def forward(self, x):
        x = self.features(x)
        self.feature_maps['layer2'] = x
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

class Model_Alternative(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 64, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool2d(2)
        )
        with torch.no_grad():
            self.feat_size = self.features(torch.zeros(1, 1, 28, 28)).numel()
        self.fc1 = nn.Linear(self.feat_size, 256)
        self.fc2 = nn.Linear(256, 10)
        self.relu = nn.ReLU()
        self.feature_maps = {}

    def forward(self, x):
        x = self.features(x)
        self.feature_maps['layer2'] = x
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# ----------------------------
# 4. Обучение чистой модели
# ----------------------------
def train_clean(model, loader, epochs, lr):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        running_loss = 0.0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

# ----------------------------
# 5. Извлечение активаций
# ----------------------------
def extract_features_and_labels(model, loader):
    model.eval()
    feats, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            _ = model(X)
            feats.append(model.feature_maps['layer2'].cpu().numpy().reshape(X.size(0), -1))
            labels.append(y.numpy().astype(int))
    return np.vstack(feats), np.concatenate(labels)

# ----------------------------
# 6. Детекторы
# ----------------------------
class MahalanobisDetector:
    def __init__(self, reg_cov=1e-4):
        self.class_means = {}
        self.inv_covs = {}
        self.reg = reg_cov
    def fit(self, X, y, num_classes=10):
        for c in range(num_classes):
            mask = y == c
            feats = X[mask]
            if len(feats) == 0: continue
            self.class_means[c] = feats.mean(axis=0)
            cov = np.cov(feats, rowvar=False)
            cov += self.reg * np.eye(cov.shape[0])
            self.inv_covs[c] = inv(cov)
        return self
    def score(self, X, y):
        scores = []
        for i in range(len(y)):
            c = int(y[i])
            if c not in self.class_means:
                scores.append(1e6); continue
            diff = X[i] - self.class_means[c]
            d = mahalanobis(diff, np.zeros_like(diff), self.inv_covs[c])
            scores.append(d**2 if np.isfinite(d) else 1e6)
        return np.array(scores)

class PCADetector:
    def __init__(self, n_components=0.95):
        self.pca = PCA(n_components=n_components, svd_solver='full')
    def fit(self, X):
        self.pca.fit(X)
        return self
    def score(self, X):
        X_red = self.pca.transform(X)
        X_rec = self.pca.inverse_transform(X_red)
        return np.sum((X - X_rec) ** 2, axis=1)

class OCSVMDetector:
    def __init__(self, nu=0.01):
        self.clf = OneClassSVM(nu=nu, gamma='scale')
    def fit(self, X):
        self.clf.fit(X)
        return self
    def score(self, X):
        return -self.clf.decision_function(X)

class IFDetector:
    def __init__(self, n_estimators=100):
        self.clf = IsolationForest(n_estimators=n_estimators, contamination='auto', random_state=42, n_jobs=-1)
    def fit(self, X):
        self.clf.fit(X)
        return self
    def score(self, X):
        return -self.clf.decision_function(X)

# ----------------------------
# 7. Метрика (только TPR@FPR)
# ----------------------------
def tpr_at_fpr(y_true, y_scores, fpr_target):
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    return np.interp(fpr_target, fpr, tpr)

# ----------------------------
# 8. Главный цикл оценки
# ----------------------------
if __name__ == "__main__":
    models = {"Model_Best": Model_Best, "Model_Alternative": Model_Alternative}
    detectors = {
        "Mahalanobis": MahalanobisDetector(),
        "PCA": PCADetector(),
        "OCSVM": OCSVMDetector(),
        "IF": IFDetector()
    }

    print(f"\n🚀 Запуск оценки атаки BIM (eps={EPS_BIM}) на чистых моделях")
    print(f"{'Модель':<20} | {'Детектор':<15} | {'TPR@FPR=1 %':<10}")
    print("-" * 50)

    for model_name, model_cls in models.items():
        print(f"  [Обучение] {model_name}...")
        model = model_cls().to(device)
        train_clean(model, trainloader, EPOCHS, LR)

        X_clean, y_clean = extract_features_and_labels(model, testloader)

        for det_name, det in detectors.items():
            if det_name == "Mahalanobis":
                det.fit(X_clean, y_clean)
                scores_clean = det.score(X_clean, y_clean)
            else:
                det.fit(X_clean)
                scores_clean = det.score(X_clean)

            # Генерация BIM-возмущений
            model.eval()
            X_adv_list, y_adv_list = [], []
            atk = torchattacks.BIM(model, eps=EPS_BIM, steps=BIM_STEPS, alpha=BIM_ALPHA)
            
            for X, y in testloader:
                X, y = X.to(device), y.to(device)
                X_adv = atk(X, y)  # Требует вычисления градиентов
                with torch.no_grad():
                    _ = model(X_adv)
                    f = model.feature_maps['layer2'].cpu().numpy().reshape(X_adv.size(0), -1)
                X_adv_list.append(f)
                y_adv_list.append(y.cpu().numpy().astype(int))
            X_adv = np.vstack(X_adv_list)
            y_adv = np.concatenate(y_adv_list)

            # Оценка детектором
            if det_name == "Mahalanobis":
                scores_adv = det.score(X_adv, y_adv)
            else:
                scores_adv = det.score(X_adv)

            # Авто-коррекция направления скоров
            if np.mean(scores_clean) > np.mean(scores_adv):
                scores_clean = -scores_clean
                scores_adv = -scores_adv

            y_true = np.concatenate([np.zeros(len(scores_clean)), np.ones(len(scores_adv))])
            y_scores = np.concatenate([scores_clean, scores_adv])

            tpr1 = tpr_at_fpr(y_true, y_scores, 0.01)
            print(f"{model_name:<20} | {det_name:<15} | {tpr1:<10.3f}")


🚀 Запуск оценки атаки BIM (eps=0.2) на чистых моделях
Модель               | Детектор        | TPR@FPR=1 %
--------------------------------------------------
  [Обучение] Model_Best...
Model_Best           | Mahalanobis     | 1.000     
Model_Best           | PCA             | 0.037     
Model_Best           | OCSVM           | 0.059     
Model_Best           | IF              | 0.006     
  [Обучение] Model_Alternative...
Model_Alternative    | Mahalanobis     | 1.000     
Model_Alternative    | PCA             | 0.051     
Model_Alternative    | OCSVM           | 0.108     
Model_Alternative    | IF              | 0.011     
